# 02 – Analysis, Risk Flag and Forecast
**Hotel Booking Analytics – Capstone Project** · Junaidi Abdul Rahim (Arjun)

This notebook answers the research objectives:
1. Headline KPIs and group comparisons (RO1–RO4)
2. A **rule-based risk flag** for cancellations (RO2)
3. A **12-month forecast** with a backtest (RO5)

Input: `data/bookings.pkl` from 01_Clean.

## Step 1 – Load the clean data and add the names from the lookup tables

In [1]:
import json, warnings                    # json: save results; warnings: hide harmless messages
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')   # nicer number display

b = pd.read_pickle('../data/processed/bookings.pkl')                  # clean bookings from notebook 01
U = '../data/raw/'
hot = pd.read_csv(U + 'Hotels.csv')                      # hotel names
ms = pd.read_csv(U + 'Market Segments.csv')              # segment + channel names
ct = pd.read_csv(U + 'Customer Types.csv')               # customer type, deposit, meal names
b = (b.merge(hot, on='Hotel_ID')                         # add hotel name
      .merge(ms, on='Market_Segment_ID')                 # add market_segment, distribution_channel
      .merge(ct, on='Customer_Type_ID'))                 # add customer_type, deposit_type, meal
b.to_pickle('../data/processed/bookings_full.pkl')
print(b.shape)

(119208, 51)


## Step 2 – Headline KPIs

In [2]:
co = b[b.is_canceled == 0]                     # 'co' = checked-out (stayed) bookings only
R = {}                                         # dictionary for the headline results
R['kpi'] = dict(
    bookings=len(b),                           # total bookings
    cancel_rate=b.is_canceled.mean(),          # share cancelled (mean of 0/1)
    cancelled=int(b.is_canceled.sum()),        # number cancelled
    realised_rev=co.realised_revenue.sum(),    # money earned
    lost_rev=b.lost_revenue.sum(),             # value of cancelled bookings
    adr=co.adr.mean(),                         # average daily rate of stayed bookings
    alos=co.total_nights.mean(),               # average length of stay
    room_nights=int(co.total_nights.sum()),    # nights sold
    repeat_rate=b.is_repeated_guest.mean(),    # share of repeat guests
    lead=b.lead_time.mean(),                   # average lead time (days)
    countries=b.country.nunique(),             # number of guest countries
    start=str(b.arrival_date.min().date()),    # first arrival date
    end=str(b.arrival_date.max().date()))      # last arrival date
pd.Series(R['kpi'])

bookings               119208
cancel_rate             0.371
cancelled               44198
realised_rev   25,986,976.030
lost_rev       16,721,837.120
adr                   100.171
alos                    3.390
room_nights            254282
repeat_rate             0.031
lead                  104.109
countries                 178
start              2015-07-01
end                2017-08-31
dtype: object

## Step 3 – Compare groups
The function `grp()` gives, for any column: bookings, cancellation rate, realised and lost revenue, lead time, ADR and average stay.

In [3]:
def grp(col, order=None):
    # main numbers for every value of 'col'
    g = b.groupby(col).agg(bookings=('Booking_ID', 'size'),          # count bookings
                           cancel_rate=('is_canceled', 'mean'),      # share cancelled
                           realised_rev=('realised_revenue', 'sum'), # money earned
                           lost_rev=('lost_revenue', 'sum'),         # money lost
                           lead=('lead_time', 'mean')).reset_index() # average lead time
    g2 = co.groupby(col).agg(adr=('adr', 'mean'),                    # price and stay use stayed bookings only
                             alos=('total_nights', 'mean')).reset_index()
    g = g.merge(g2, on=col, how='left')                              # put both together
    if order:                                                        # keep a logical order (e.g. lead-time bands)
        g[col] = pd.Categorical(g[col], order, ordered=True)
        g = g.sort_values(col)
    else:                                                            # otherwise biggest first
        g = g.sort_values('bookings', ascending=False)
    g[col] = g[col].astype(str)
    return g

T = {}                                                               # dictionary of result tables
T['hotel'] = grp('hotel')
T['segment'] = grp('market_segment')
T['channel'] = grp('distribution_channel')
T['deposit'] = grp('deposit_type')
T['customer'] = grp('customer_type')
T['meal'] = grp('meal')
T['lead_band'] = grp('lead_time_band', ['0-7 days', '8-30 days', '31-90 days', '91-180 days', '181-365 days', '365+ days'])
T['stay_band'] = grp('stay_length_band', ['Day use (0)', '1-3 nights', '4-7 nights', '8-14 nights', '15+ nights'])
T['repeat'] = grp('is_repeated_guest')
T['prev_cancel'] = grp('has_previous_cancel')
T['special_req'] = grp('total_of_special_requests').sort_values('total_of_special_requests')
T['parking'] = grp('required_car_parking_spaces').sort_values('required_car_parking_spaces')
T['country'] = grp('country_name').head(10)                          # top 10 countries
T['season'] = grp('season', ['Spring (Mar-May)', 'Summer (Jun-Aug)', 'Autumn (Sep-Nov)', 'Winter (Dec-Feb)'])
T['hotel']

,hotel,bookings,cancel_rate,realised_rev,lost_rev,lead,adr,alos
0,City Hotel,79162,0.418,"14,385,342.000","10,879,659.780",109.885,106.036,2.918
1,Resort Hotel,40046,0.278,"11,601,634.030","5,842,177.340",92.691,90.826,4.142


In [4]:
T['lead_band']          # cancellation rises with lead time: 9.6% -> 67.7%

,lead_time_band,bookings,cancel_rate,realised_rev,lost_rev,lead,adr,alos
0,0-7 days,19635,0.096,"3,291,878.630","449,424.930",2.325,89.705,1.965
4,8-30 days,18945,0.279,"4,338,994.340","2,045,797.070",17.944,104.684,3.005
2,31-90 days,29528,0.377,"7,004,232.820","4,321,162.780",57.312,103.966,3.646
5,91-180 days,26420,0.447,"6,768,390.410","5,130,169.340",132.109,106.265,4.357
1,181-365 days,21533,0.555,"4,295,543.550","4,419,447.230",253.391,96.776,4.646
3,365+ days,3147,0.677,"287,936.280","355,835.770",440.462,98.101,3.072


In [5]:
T['segment']            # Groups 61%, Online TA 37%, Direct 15%

,market_segment,bookings,cancel_rate,realised_rev,lost_rev,lead,adr,alos
6,Online TA,56408,0.368,"13,707,139.090","10,227,646.110",83.066,114.035,3.328
5,Offline TA/TO,24181,0.343,"5,659,291.940","2,486,958.150",135.101,83.555,4.266
4,Groups,19790,0.611,"1,869,129.520","2,800,543.980",187.090,77.305,3.225
3,Direct,12582,0.154,"4,097,842.400","993,409.820",49.898,114.095,3.072
2,Corporate,5282,0.188,"577,892.190","196,383.070",22.140,67.291,1.983
1,Complementary,728,0.122,"4,812.530",269.990,13.552,3.112,1.700
0,Aviation,235,0.221,"70,868.360","16,578.000",4.472,100.569,3.781
7,Undefined,2,1.000,0.000,48.000,1.500,NaN,NaN


In [6]:
T['special_req']        # guests with requests cancel less

,total_of_special_requests,bookings,cancel_rate,realised_rev,lost_rev,lead,adr,alos
0,0,70199,0.478,"10,874,478.580","11,517,947.670",114.411,90.010,3.240
1,1,33183,0.220,"9,544,474.100","3,535,286.880",89.005,105.594,3.455
2,2,12952,0.221,"4,343,189.190","1,425,935.200",89.010,117.210,3.621
3,3,2494,0.178,"1,022,519.900","222,609.180",92.626,124.435,3.960
4,4,340,0.106,"179,641.170","18,873.190",110.521,131.902,4.319
5,5,40,0.050,"22,673.090","1,185.000",105.550,128.233,4.632


## Step 4 – Monthly tables (for trends and the forecast)

In [7]:
b['ym'] = b.arrival_date.dt.to_period('M').astype(str)               # '2015-07', '2015-08', ...
mh = (b.groupby(['ym', 'hotel'])
       .agg(bookings=('Booking_ID', 'size'),                         # bookings per month and hotel
            cancelled=('is_canceled', 'sum'),                        # cancelled bookings
            realised_rev=('realised_revenue', 'sum'))                # revenue
       .reset_index())
nights = co.assign(ym=co.arrival_date.dt.to_period('M').astype(str)).groupby(['ym', 'hotel']).total_nights.sum()
mh['room_nights'] = mh.set_index(['ym', 'hotel']).index.map(nights).fillna(0).values   # nights sold (stayed only)
mh['cancel_rate'] = mh.cancelled / mh.bookings
T['monthly'] = mh

T['moy'] = (co.groupby(['arrival_month_num', 'hotel'])                # ADR by month of year (seasonality)
              .agg(adr=('adr', 'mean'), bookings=('Booking_ID', 'size')).reset_index())
T['moy'].pivot(index='arrival_month_num', columns='hotel', values='adr')   # Resort: about 49 in Jan -> 181 in Aug

hotel,City Hotel,Resort Hotel
arrival_month_num,,
1,82.331,48.761
2,86.520,54.147
3,90.659,57.082
4,111.962,75.868
5,120.670,76.658
6,117.874,107.975
7,115.818,150.123
8,118.675,181.206
9,112.777,96.417


## Step 5 – Rule-based cancellation risk flag
Built from the findings above (no machine learning), 1 point each:
1. Lead time over 90 days (cancel rate rises with lead time)
2. Groups or Online TA segment (highest-cancelling segments)
3. No special requests (engaged guests cancel less)

**0–1 points = Low · 2 = Medium · 3 = High**

In [8]:
b['risk_factors'] = ((b.lead_time > 90).astype(int)                                     # point 1
                     + b.market_segment.isin(['Groups', 'Online TA']).astype(int)      # point 2
                     + (b.total_of_special_requests == 0).astype(int))                 # point 3
b['risk_flag'] = b.risk_factors.map({0: 'Low', 1: 'Low', 2: 'Medium', 3: 'High'})       # points -> label

T['risk_flag'] = (b.groupby('risk_flag')
                   .agg(bookings=('Booking_ID', 'size'),
                        actual_cancel_rate=('is_canceled', 'mean'),     # did they really cancel?
                        cancelled=('is_canceled', 'sum'),
                        lost_rev=('lost_revenue', 'sum'),
                        realised_rev=('realised_revenue', 'sum'))
                   .reindex(['Low', 'Medium', 'High']).reset_index())
T['risk_factors'] = b.groupby('risk_factors').agg(bookings=('Booking_ID', 'size'),
                                                  actual_cancel_rate=('is_canceled', 'mean'),
                                                  lost_rev=('lost_revenue', 'sum')).reset_index()
T['risk_flag_hotel'] = b.groupby(['hotel', 'risk_flag']).agg(bookings=('Booking_ID', 'size'),
                                                             actual_cancel_rate=('is_canceled', 'mean'),
                                                             lost_rev=('lost_revenue', 'sum')).reset_index()
b[['Booking_ID', 'risk_factors', 'risk_flag']].to_csv('../data/processed/Fact_Risk_Flag.csv', index=False)
T['risk_flag']          # High about 72% vs Low about 19%: the rule works

,risk_flag,bookings,actual_cancel_rate,cancelled,lost_rev,realised_rev
0,Low,54299,0.186,10113,"3,841,580.160","14,208,484.570"
1,Medium,44482,0.436,19390,"7,749,994.040","9,771,340.840"
2,High,20427,0.719,14695,"5,130,262.920","2,007,150.620"


## Step 6 – Forecast with backtest
**Method:** each month = same month last year × damped growth.
Damped growth = 1 + 0.5 × (last 6 months ÷ same 6 months a year earlier − 1).

**Backtest:** pretend it is February 2017, forecast March–August 2017, and compare with what really happened (MAPE = average % error). A plain "same as last year" forecast is the benchmark.

In [9]:
def sn_growth(y, end, h):
    # y = monthly series, end = last month we 'know', h = number of months to forecast
    hist = y[:end]                                   # history up to the end month
    last6 = hist.iloc[-6:].sum()                     # last 6 months
    prev6 = hist.iloc[-18:-12].sum()                 # the same 6 months a year earlier
    g = 1 + 0.5 * (last6 / prev6 - 1)                # damped growth: half of the recent growth
    idx = pd.date_range(pd.Timestamp(end) + pd.offsets.MonthBegin(1), periods=h, freq='MS')   # future months
    return pd.Series([y[d - pd.DateOffset(years=1)] * g for d in idx], idx), g                 # last year x growth

fc = []                                              # rows of the forecast table
for h in ['City Hotel', 'Resort Hotel']:
    s = mh[mh.hotel == h].set_index('ym')
    for metric in ['bookings', 'realised_rev']:
        y = s[metric].astype(float)
        y.index = pd.PeriodIndex(y.index, freq='M').to_timestamp()       # month labels -> dates
        yte = y['2017-03':]                                              # the 'test' months Mar-Aug 2017
        bt, _ = sn_growth(y, '2017-02-01', 6)                            # backtest forecast
        mape = float((abs(bt.values - yte.values) / yte.values).mean())  # our error
        naive = pd.Series([y[d - pd.DateOffset(years=1)] for d in yte.index], yte.index)        # same as last year
        nmape = float((abs(naive.values - yte.values) / yte.values).mean())                     # benchmark error
        f, g = sn_growth(y, '2017-08-01', 12)                            # real forecast: Sep 2017 - Aug 2018
        for d, v in y.items():  fc.append(dict(hotel=h, metric=metric, month=d.strftime('%Y-%m'), type='Actual',   value=v, mape=None, naive_mape=None, growth=None))
        for d, v in bt.items(): fc.append(dict(hotel=h, metric=metric, month=d.strftime('%Y-%m'), type='Backtest', value=v, mape=mape, naive_mape=nmape, growth=None))
        for d, v in f.items():  fc.append(dict(hotel=h, metric=metric, month=d.strftime('%Y-%m'), type='Forecast', value=v, mape=mape, naive_mape=nmape, growth=g))
T['forecast'] = pd.DataFrame(fc)

F = T['forecast']
summary = F.groupby(['hotel', 'metric']).agg(mape=('mape', 'max'), naive_mape=('naive_mape', 'max'), growth=('growth', 'max'))
summary['forecast_total'] = F[F.type == 'Forecast'].groupby(['hotel', 'metric']).value.sum()
summary                 # revenue MAPE about 6-8% vs naive 15-18%

mape  naive_mape  growth  forecast_total
hotel        metric                                                
City Hotel   bookings     0.114       0.100   1.044      42,896.246
             realised_rev 0.084       0.179   1.110   8,913,492.340
Resort Hotel bookings     0.121       0.111   1.017      19,724.562
             realised_rev 0.061       0.155   1.085   5,947,516.105

## Step 7 – Save the results for the charts notebook

In [10]:
b.to_pickle('../data/processed/bookings_flag.pkl')                                 # bookings with the risk flag
pd.to_pickle(T, '../data/processed/tables.pkl')                                    # all result tables
json.dump(R, open('../data/processed/results.json', 'w'), indent=1, default=float) # headline KPIs
print('Saved. Total forecast revenue:', round(F[(F.type == 'Forecast') & (F.metric == 'realised_rev')].value.sum() / 1e6, 2), 'million euros')

Saved. Total forecast revenue: 14.86 million euros


### Summary
- 37.1% of bookings cancelled; €26.0M earned, €16.7M lost.
- Cancellation rises with lead time (9.6% → 67.7%); Groups 61%, Online TA 37%, Direct 15%.
- Risk flag: High 71.9% vs Low 18.6% actually cancelled.
- Forecast Sep 2017 – Aug 2018: about **€14.86M (+10%)**, backtest error 6–8%.
- Next: run **03_Charts.ipynb**.